# Teaching Stable Diffusion an Interior Style with LoRA

**Goal.** Teach Stable Diffusion 1.5 a specific interior style (here: **Japandi**, a mix of Japanese and Scandinavian design) from only ~20 photos, then **measure** how well it learned the style.

### What is LoRA?
A model like Stable Diffusion has huge weight matrices. Fine-tuning all of them needs a lot of memory and gives a 4 GB file.
**LoRA (Low-Rank Adaptation)** freezes the original model and learns a small *update* for some layers:

$$W_{new} = W + B \cdot A$$

If $W$ is 320×320 (about 102,000 numbers), $A$ is 8×320 and $B$ is 320×8, so the update has only about 5,000 numbers. The `8` is the **rank**: higher rank = more capacity, bigger file.
Result: training fits on a free T4 GPU, and the LoRA file is only a few MB. You can switch it on and off, and choose how strong it is.

### Plan
1. Train a LoRA on ~20 Japandi interior photos with a trigger word (`jpdi`).
2. Generate the same prompts **without** and **with** the LoRA (strength 0, 0.5, 1.0).
3. Measure with CLIP:
   - **Style similarity**: how close the images are to the training photos (higher = style learned)
   - **Prompt match**: is a bedroom still a bedroom? (should stay high)
   - **Memorization**: is the model just copying a training photo? (very high = bad)
4. Bonus: apply the LoRA to my own room photos with ControlNet, keeping the geometry.

## 1. Setup

In [ ]:
!pip uninstall -y torchao
!pip install -q diffusers transformers accelerate peft opencv-python

## 2. Check the GPU (must print True and Tesla T4)

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Upload the training photos
Upload **15–25 photos** of the same style, either as separate images or as one `.zip` file.
- **Colab:** a file picker opens when you run the cell.
- **Kaggle:** first click *Add Input → Upload* and upload the photos as a dataset. The cell finds them in `/kaggle/input`. Put your own room photo in the same dataset with a name starting with `my_room` (for section 11).
Good training data: one consistent style, different rooms and angles, no text or watermarks.

In [ ]:
import os, io, glob, zipfile
import numpy as np
from PIL import Image
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
IN_COLAB = "google.colab" in str(get_ipython())
os.makedirs("train_images", exist_ok=True)

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(data)).extractall("train_images")
        else:
            open(os.path.join("train_images", name), "wb").write(data)
    src = "train_images"
else:  # Kaggle: photos come from the uploaded dataset
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("train_images")
    src = "/kaggle/input"

paths = sorted(p for p in glob.glob(f"{src}/**/*", recursive=True) + glob.glob("train_images/**/*", recursive=True)
               if p.lower().endswith(IMG_EXT) and not os.path.basename(p).lower().startswith("my_room"))
paths = sorted(set(paths))
print(len(paths), "training images")

def show_grid(imgs, size=192, cols=8):
    rows = (len(imgs) + cols - 1) // cols
    grid = Image.new("RGB", (cols * size, rows * size), "white")
    for i, im in enumerate(imgs):
        grid.paste(im.convert("RGB").resize((size, size)), ((i % cols) * size, (i // cols) * size))
    display(grid)
    return grid

show_grid([Image.open(p) for p in paths]);

## 4. Settings

In [ ]:
BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
TRIGGER = "jpdi"                                   # a new word the model will connect to the style
ROOM_WORDS = {"living": "living room", "bedroom": "bedroom", "dining": "dining room",
              "kitchen": "kitchen", "detail": "interior detail"}
def caption_for(path):
    # the room type comes from the file name (living_01.jpg -> "living room"); unknown names -> "interior"
    prefix = os.path.basename(path).split("_")[0].lower()
    return f"a photo of a {TRIGGER} style {ROOM_WORDS.get(prefix, 'interior')}"
RES = 512                                          # SD 1.5 was trained on 512x512
RANK = 8                                           # LoRA rank (capacity)
LR = 1e-4                                          # learning rate
STEPS = 800                                        # about 45 passes over 17 images
SEED = 42

## 5. Prepare the data
Each photo is cropped to a square, resized to 512×512 and turned into a **latent** with the VAE (the compressed 64×64 form that Stable Diffusion works with). A flipped copy is added for variety.
Each photo gets its own caption with the room type from its file name, for example *"a photo of a jpdi style bedroom"*. This way the room word explains **what** is in the photo, and the trigger word `jpdi` only has to learn the **style**. The captions are turned into text embeddings once. After that, the VAE and the text encoder are removed from the GPU to free memory.

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL

def to_tensor(p):
    img = Image.open(p).convert("RGB")
    w, h = img.size; s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((RES, RES), Image.BICUBIC)
    return torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 127.5 - 1   # values in [-1, 1]

vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float16).to("cuda")
tokenizer = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda")

latents, text_embs = [], []
with torch.no_grad():
    for p in paths:
        ids = tokenizer(caption_for(p), padding="max_length", max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt").input_ids.cuda()
        emb = text_encoder(ids)[0]
        x = to_tensor(p)
        for img in (x, torch.flip(x, dims=[2])):                     # original + horizontal flip
            lat = vae.encode(img.unsqueeze(0).half().cuda()).latent_dist.sample()
            latents.append(lat * vae.config.scaling_factor)
            text_embs.append(emb)

for p in paths:
    print(f"{os.path.basename(p):18s} -> {caption_for(p)}")
print(len(latents), "latents of shape", tuple(latents[0].shape))
del vae, text_encoder; torch.cuda.empty_cache()

## 6. Add LoRA to the UNet
The UNet is the part of Stable Diffusion that removes noise. It stays **frozen**; LoRA adds small trainable matrices to its attention layers (`to_q`, `to_k`, `to_v`, `to_out`).

In [ ]:
from diffusers import UNet2DConditionModel, DDPMScheduler
from diffusers.training_utils import cast_training_params
from peft import LoraConfig

unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16).to("cuda")
unet.requires_grad_(False)
unet.add_adapter(LoraConfig(r=RANK, lora_alpha=RANK, init_lora_weights="gaussian",
                            target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
cast_training_params(unet, dtype=torch.float32)      # LoRA weights in full precision for stable training
unet.enable_gradient_checkpointing()                 # saves memory

trainable = [p for p in unet.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable); n_all = sum(p.numel() for p in unet.parameters())
print(f"Trainable: {n_train:,} of {n_all:,} parameters ({100 * n_train / n_all:.2f}%)")

## 7. Train
Each step: take a training latent, add random noise at a random strength, and ask the UNet to predict that noise. The loss is how wrong the prediction is. Only the LoRA weights change.
Takes about 10 minutes on a T4.

In [ ]:
import time
import torch.nn.functional as F

noise_scheduler = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-2)
scaler = torch.amp.GradScaler("cuda")
gen = torch.Generator().manual_seed(SEED)
torch.manual_seed(SEED)

losses, t0 = [], time.time()
unet.train()
for step in range(1, STEPS + 1):
    i = torch.randint(len(latents), (1,), generator=gen).item()
    lat, text_emb = latents[i], text_embs[i]
    noise = torch.randn_like(lat)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device="cuda").long()
    noisy = noise_scheduler.add_noise(lat, noise, t)

    with torch.autocast("cuda", dtype=torch.float16):
        pred = unet(noisy, t, text_emb).sample
    loss = F.mse_loss(pred.float(), noise.float())

    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f"step {step}/{STEPS}  loss {np.mean(losses[-100:]):.4f}  ({time.time() - t0:.0f} s)")

print(f"Training time: {(time.time() - t0) / 60:.1f} min, peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
smooth = np.convolve(losses, np.ones(50) / 50, mode="valid")
plt.figure(figsize=(7, 3))
plt.plot(losses, alpha=0.25, label="loss per step")
plt.plot(range(49, len(losses)), smooth, label="moving average (50 steps)")
plt.xlabel("step"); plt.ylabel("MSE loss"); plt.legend(); plt.title("LoRA training loss")
plt.tight_layout(); plt.savefig("lora_loss.png", dpi=150); plt.show()

*Note: diffusion loss is very noisy, because every step uses a random noise level. A flat or slowly falling average is normal; the real test is the images.*

## 8. Save the LoRA

In [ ]:
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers
from diffusers import StableDiffusionPipeline

lora_state = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
StableDiffusionPipeline.save_lora_weights("jpdi_lora", unet_lora_layers=lora_state, safe_serialization=True)
size_mb = sum(os.path.getsize(f) for f in glob.glob("jpdi_lora/*")) / 1e6
print(f"LoRA saved: {size_mb:.1f} MB (the full model is about 4,000 MB)")

del unet, optimizer; torch.cuda.empty_cache()

## 9. Without vs with LoRA
Same prompts and seeds, LoRA strength 0 (off), 0.5 and 1.0. Two room types (bathroom, office) may not be in the training photos: they test if the style **transfers** to new rooms.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.load_lora_weights("jpdi_lora")

rooms = ["living room", "bedroom", "kitchen", "bathroom", "home office"]
scales = [0.0, 0.5, 1.0]
negative = "blurry, distorted, low quality, deformed, text, watermark"
generated = {}   # (room, scale) -> image

for room in rooms:
    row = []
    for s in scales:
        img = pipe(f"a photo of a {TRIGGER} style {room}", negative_prompt=negative,
                   num_inference_steps=25, guidance_scale=7.5,
                   cross_attention_kwargs={"scale": s},
                   generator=torch.manual_seed(SEED)).images[0]
        generated[(room, s)] = img
        img.save(f"gen_{room.replace(' ', '_')}_scale{s}.jpg", quality=90)
        row.append(img)
    print(f"{room}: LoRA strength 0 | 0.5 | 1.0")
    show_grid(row, size=320, cols=3).save(f"compare_{room.replace(' ', '_')}.jpg", quality=90)

## 10. Measure with CLIP
CLIP turns images and texts into vectors; similar things get similar vectors.
- **Style similarity**: cosine between a generated image and the *average* of the training photos. Higher = closer to the style.
- **Prompt match**: cosine between the image and "a photo of a {room}". It should stay high, so a bedroom is still a bedroom.
- **Memorization**: cosine to the *most similar single* training photo. Around 0.95 or more would mean the model is copying.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
import pandas as pd

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def img_emb(imgs):
    x = clip_proc(images=imgs, return_tensors="pt").to("cuda")
    e = clip.visual_projection(clip.vision_model(pixel_values=x["pixel_values"]).pooler_output)
    return e / e.norm(dim=-1, keepdim=True)

@torch.no_grad()
def txt_emb(texts):
    x = clip_proc(text=texts, return_tensors="pt", padding=True).to("cuda")
    e = clip.text_projection(clip.text_model(input_ids=x["input_ids"],
                                             attention_mask=x["attention_mask"]).pooler_output)
    return e / e.norm(dim=-1, keepdim=True)

train_e = img_emb([Image.open(p).convert("RGB") for p in paths])
style_center = train_e.mean(0); style_center = style_center / style_center.norm()

rows = []
for (room, s), img in generated.items():
    e = img_emb([img])[0]
    rows.append({"room": room, "LoRA strength": s,
                 "Style similarity ↑": float(e @ style_center),
                 "Prompt match ↑": float(e @ txt_emb([f"a photo of a {room}"])[0]),
                 "Memorization (max sim to a training photo) ↓": float((train_e @ e).max())})

df = pd.DataFrame(rows)
display(df.round(3))
summary = df.groupby("LoRA strength").mean(numeric_only=True).round(3)
display(summary)

**What I saw (fill in after running).**
- Without LoRA (strength 0): …
- With LoRA (strength 1.0): …
- New rooms (bathroom, office): …

**Result.** …

## 11. Bonus: restyle my own room, keeping the geometry
Combine the LoRA with what I learned in the render project: Canny ControlNet + img2img keeps the walls and furniture in place, while the LoRA brings the Japandi style.
Upload one of your own room photos.

In [ ]:
import cv2
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel

controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe_cn = StableDiffusionControlNetImg2ImgPipeline(**{**pipe.components, "controlnet": controlnet.to("cuda")})

if IN_COLAB:
    room_file = list(files.upload())[0]
else:
    room_file = [p for p in glob.glob("/kaggle/input/**/*", recursive=True)
                 if os.path.basename(p).lower().startswith("my_room")][0]
W, H = 768, 432
room = Image.open(room_file).convert("RGB").resize((W, H))
edges = Image.fromarray(np.stack([cv2.Canny(np.array(room), 100, 200)] * 3, axis=-1))

outs = []
for s in [0.0, 1.0]:
    out = pipe_cn(f"a photo of a {TRIGGER} style living room", negative_prompt=negative,
                  image=room, control_image=edges, strength=0.6, num_inference_steps=25,
                  cross_attention_kwargs={"scale": s}, generator=torch.manual_seed(SEED)).images[0]
    outs.append(out)

row = Image.new("RGB", (W * 3, H))
for i, im in enumerate([room] + outs):
    row.paste(im, (i * W, 0))
row.save("restyle_my_room.jpg", quality=90)
display(row)   # original | without LoRA | with LoRA

**What I saw (fill in after running).** …

## 12. Download the results

In [ ]:
df.to_csv("lora_metrics.csv", index=False)
!zip -q -r lora_results.zip jpdi_lora compare_*.jpg restyle_my_room.jpg lora_loss.png lora_metrics.csv
if IN_COLAB:
    files.download("lora_results.zip")
else:
    print("Kaggle: download lora_results.zip from the Output panel on the right (/kaggle/working)")

*Optional: share the LoRA on the Hugging Face Hub*
```python
from huggingface_hub import notebook_login, create_repo, upload_folder
notebook_login()
create_repo("Mervecaliskan/japandi-interior-lora", exist_ok=True)
upload_folder(repo_id="Mervecaliskan/japandi-interior-lora", folder_path="jpdi_lora")
```